In [1]:
import boto3
import os
from time import gmtime, strftime
import sagemaker
from sagemaker import get_execution_role

/opt/conda/lib/python3.11/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "json" in "MonitoringDatasetFormat" shadows an attribute in parent "Base"
  warnings.warn(


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
# Setup
client = boto3.client(service_name="sagemaker")

# runtime = boto3.client(service_name="sagemaker-runtime")
boto_session = boto3.session.Session()
s3 = boto_session.resource('s3')
region = boto_session.region_name
print(region)
sagemaker_session = sagemaker.Session()
role = get_execution_role()

us-east-1


In [3]:
# Endpoint container
custom_image_uri = os.getenv("SP_CUSTOM_XGBOOST_IMAGE_URI", "719805443631.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost-custom:1.7-7")

In [4]:
custom_image_uri

'719805443631.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost-custom:1.7-7'

In [5]:
if custom_image_uri:
    container_uri = custom_image_uri
else:
    container_uri = sagemaker.image_uris.retrieve(framework="xgboost",
                                                  region=region,
                                                  version="1.7-1")

In [6]:
container_uri

'719805443631.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost-custom:1.7-7'

In [7]:
default_bucket = sagemaker_session.default_bucket()
prefix = 'fernando_test'
modelprefix = prefix + '/model/'
key = modelprefix + "model.tar.gz"
model_artifacts = f"s3://{default_bucket}/{key}"
print(model_artifacts)

s3://sagemaker-us-east-1-719805443631/fernando_test/model/model.tar.gz


In [8]:
# Model
# model_name = "oejr-prob-sklearn-1.4" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
model_name = "oejr-prob-sklearn-" + strftime("%Y%m%d%H%M%S", gmtime())
print("Model name: " + model_name)
create_model_response = client.create_model(
    ModelName=model_name,
    Containers=[
        {
            "Image": container_uri,
            "Mode": "SingleModel",
            "ModelDataUrl": model_artifacts,
            "Environment": {
                "SAGEMAKER_PROGRAM": "inference.py",
                "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/program"
            } if custom_image_uri else {}
        }
    ],
    ExecutionRoleArn=role,
)
print("Model Arn: " + create_model_response["ModelArn"])

Model name: oejr-prob-sklearn-20260506151716
Model Arn: arn:aws:sagemaker:us-east-1:719805443631:model/oejr-prob-sklearn-20260506151716


In [9]:
# Endpoint configuration
endpoint_configuration_name = "oejr-prob-epc" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
endpoint_config_response = client.create_endpoint_config(
    EndpointConfigName=endpoint_configuration_name,
    ProductionVariants=[
        {
            "VariantName": "allmodels",
            "ModelName": model_name,
            "InstanceType": "ml.m5.large",
            "InitialInstanceCount": 1
        },
    ],
)
print("Endpoint Configuration Arn: " + endpoint_config_response["EndpointConfigArn"])

Endpoint Configuration Arn: arn:aws:sagemaker:us-east-1:719805443631:endpoint-config/oejr-prob-epc2026-05-06-15-17-45


In [10]:
# Endpoint
endpoint_name = "oejr-prob-ep" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
create_endpoint_response = client.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=endpoint_configuration_name,
)
print("Endpoint Arn: " + create_endpoint_response["EndpointArn"])
print("Creating endpoint...")

Endpoint Arn: arn:aws:sagemaker:us-east-1:719805443631:endpoint/oejr-prob-ep2026-05-06-15-17-47
Creating endpoint...
